# Pipeline 04a: deterministic template baseline

Purpose: a reviewer baseline. It answers the question:
what does the LLM add on top of a rule based template generator?

Same inputs as the LLM pipelines (local SHAP/EBM JSONs) and the same three part
structure [VORHERSAGE] / [TREIBER] / [EMPFEHLUNG], fully deterministic.

Expected picture in the LLM judge (NB 05):
* Faithfulness: the template names the exact top 3 drivers, so a high score is expected
* Clarity: no contextual rewording, no narrative flow, so lower than the LLM
* Completeness: structure correct, but the recommendation is purely threshold based without context

In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from utils import INSTANCE_IDS, RESULTS_DIR, EXPLANATIONS_DIR

LOSS_KEY   = 'poisson_log'
XAI_MODELS = ['xgb', 'ebm']

# n=20 validity run: the 10 stratified instances, 1 generation each (the template
# is DETERMINISTIC, so there is no variance to measure and no _gen suffix).
GEN_INSTANCE_IDS = INSTANCE_IDS
OUT_DIR          = RESULTS_DIR / 'pipeline00'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Instances: {len(GEN_INSTANCE_IDS)}  (1 generation, deterministic)')
print(f'Models:    {XAI_MODELS}')
print(f'Output:    {OUT_DIR}')

In [ ]:
# Same mappings as build_judge_prompt in NB 05.
WEEKDAYS = {
    0: 'Sunday', 1: 'Monday', 2: 'Tuesday', 3: 'Wednesday',
    4: 'Thursday', 5: 'Friday', 6: 'Saturday',
}
MONTHS = {
    1: 'January', 2: 'February', 3: 'March', 4: 'April',
    5: 'May', 6: 'June', 7: 'July', 8: 'August',
    9: 'September', 10: 'October', 11: 'November', 12: 'December',
}
WEATHER = {
    1: 'clear/few clouds',
    2: 'mist/cloudy',
    3: 'light rain/snow',
    4: 'heavy rain/thunderstorm',
}
FEATURE_NAMES = {
    'hr':         'hour',
    'temp':       'temperature',
    'hum':        'humidity',
    'windspeed':  'wind speed',
    'yr':         'year',
    'mnth':       'month',
    'weekday':    'weekday',
    'weathersit': 'weather',
    'holiday':    'holiday status',
}


def readable_value(feature: str, raw: float) -> str:
    """Denormalise a raw feature value into a readable string."""
    if feature == 'hr':
        return f'{int(raw):02d}:00'
    elif feature == 'temp':
        return f'~{raw * 41:.1f} C'
    elif feature == 'hum':
        return f'{raw * 100:.0f} %'
    elif feature == 'windspeed':
        return f'{raw * 67:.1f} km/h'
    elif feature == 'yr':
        return '2011' if int(raw) == 0 else '2012'
    elif feature == 'mnth':
        return MONTHS.get(int(raw), str(int(raw)))
    elif feature == 'weekday':
        return WEEKDAYS.get(int(raw), str(int(raw)))
    elif feature == 'weathersit':
        return WEATHER.get(int(raw), str(int(raw)))
    elif feature == 'holiday':
        return 'holiday' if int(raw) == 1 else 'no holiday'
    else:
        return str(raw)


print('Helper tables loaded.')

In [ ]:
def generate_template(data: dict) -> str:
    """Deterministic template from a local explanation JSON.
    Structure identical to the LLM pipelines: [PREDICTION] / [DRIVERS] / [RECOMMENDATION]."""
    pred   = data['prediction']
    actual = data['y_true']
    fv     = data['feature_values']
    top3   = data['contributions'][:3]

    hour    = int(fv['hr'])
    weekday = WEEKDAYS.get(int(fv['weekday']), str(fv['weekday']))
    month   = MONTHS.get(int(fv['mnth']), str(fv['mnth']))
    year    = '2011' if int(fv['yr']) == 0 else '2012'

    # [PREDICTION] section
    prediction = (
        f'[PREDICTION] For {weekday}, {month} {year}, {hour:02d}:00 the model '
        f'predicts {pred:.0f} bike rentals in total (actual value: {actual:.0f}).'
    )

    # [DRIVERS] section
    lines = []
    for c in top3:
        feat      = c['feature']
        contrib   = c['contribution']
        val_str   = readable_value(feat, c['value'])
        direction = 'raising' if contrib > 0 else 'lowering'
        fname     = FEATURE_NAMES.get(feat, feat)
        lines.append(f'* {fname} ({val_str}): {direction} (contribution: {contrib:+.2f})')
    drivers = '[DRIVERS] The three most important influencing factors are:\n' + '\n'.join(lines)

    # [RECOMMENDATION] section
    if 7 <= hour <= 9:
        daypart = 'morning commute'
    elif 17 <= hour <= 19:
        daypart = 'evening commute'
    elif 10 <= hour <= 16:
        daypart = 'daytime operation'
    else:
        daypart = 'off peak time'

    if pred >= 350:
        level  = 'High demand'
        action = 'Provide enough bikes and staff at busy stations.'
    elif pred >= 150:
        level  = 'Moderate demand'
        action = 'Keep regular operational readiness.'
    else:
        level  = 'Low demand'
        action = 'Resources can be reduced in a targeted way.'

    recommendation = f'[RECOMMENDATION] {level} during the {daypart}. {action}'

    return '\n\n'.join([prediction, drivers, recommendation])


# Smoke test
_test = json.loads((EXPLANATIONS_DIR / f'local_xgb_{LOSS_KEY}_inst224.json').read_text())
print(generate_template(_test))

In [ ]:
n_written = 0
for xai in XAI_MODELS:
    for iid in GEN_INSTANCE_IDS:
        src  = EXPLANATIONS_DIR / f'local_{xai}_{LOSS_KEY}_inst{iid}.json'
        data = json.loads(src.read_text())

        t0          = time.perf_counter()
        explanation = generate_template(data)
        elapsed     = round(time.perf_counter() - t0, 6)

        record = {
            'explanation':  explanation,
            'elapsed_s':    elapsed,
            'usage':        {'input_tokens': 0, 'output_tokens': 0, 'cache_read_input_tokens': 0},
            'n_tool_calls': 0,
            'y_true':       data['y_true'],
            'prediction':   data['prediction'],
        }

        # Deterministic generation, no _gen suffix (like generation_filename(n=1)).
        out_file = OUT_DIR / f'{xai}_inst{iid}.json'
        out_file.write_text(json.dumps(record, indent=2, ensure_ascii=False))
        n_written += 1
        if n_written <= 5 or n_written % 50 == 0:
            print(f'  [{n_written:3d}] {xai.upper()} inst={iid:4d}: {len(explanation.split()):3d} words  '
                  f'pred={data["prediction"]:.0f}  true={data["y_true"]:.0f}')

print(f'\nDone. {n_written} files saved in {OUT_DIR} (no API call, $0).')